In [0]:
# Imports to access functions from spark
from pyspark.sql.functions import * 
from pyspark.sql.types import *

# Silver Layer Script

## Data Access using App

#### **Important** : The information enclosed in <> in below code should be replaced with appropriate data from Azure.

In [0]:
spark.conf.set("fs.azure.account.auth.type.<datalake_name>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<datalake_name>.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<datalake_name>.dfs.core.windows.net", "<application_id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<datalake_name>.dfs.core.windows.net", "<service_credential>")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<datalake_name>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory_id>/oauth2/token")


## Data Loading

### Read Calendar Data

In [0]:
df_cal = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Calendar')

In [0]:
df_cus = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Customers')

In [0]:
df_procat = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Product_Categories')

In [0]:
df_pro = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Products')

In [0]:
df_ret = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Returns')

In [0]:
df_sales = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Sales*')

In [0]:
df_ter = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/AdventureWorks_Territories')

In [0]:
df_subcat = spark.read.format('csv')\
            .option("header", True)\
            .option("inferSchema", True)\
            .load('abfss://bronze@<datalake_name>.dfs.core.windows.net/Product_Subcategories')


## Transformations


### Calendar

In [0]:
# Display small set of data
df_cal.show(5)

In [0]:
df_cal = df_cal.withColumn('Month', month(col('Date')))\
            .withColumn('Year', year(col('Date')))

In [0]:
df_cal.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_Calendar")\
            .save()

### Cutomers

In [0]:
df_cus = df_cus.withColumn('fullName', concat_ws(' ', col('Prefix'), col('FirstName'), col('lastName')))

In [ ]:
df_cus.show(5)

In [ ]:
df_cus.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_Customers")\
            .save()

### Sub Categories

In [ ]:
df_subcat.show(5)

In [ ]:
df_subcat.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_SubCategories")\
            .save()

### Products

In [ ]:
df_pro = df_pro.withColumn('ProductSKU', split(col('ProductSKU'), '-')[0])\
                .withColumn('ProductName', split(col('ProductName'), ' ')[0])

In [ ]:
df_pro.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_Products")\
            .save()

### Returns

In [ ]:
df_ret.show(5)

In [ ]:
df_ret.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_Returns")\
            .save()

### Territories

In [ ]:
df_ter.show(5)

In [ ]:
df_ter.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_Territories")\
            .save()

### Sales

In [ ]:
df_sales = df_sales.withColumn('StockDate', to_timestamp('StockDate'))

In [ ]:
df_sales = df_sales.withColumn('OrderNumber', regexp_replace(col('OrderNumber'), 'S', 'T'))

In [ ]:
df_sales = df_sales.withColumn('multiply', col('OrderLineItem')*col('OrderQuantity'))

In [ ]:
df_sales.write.format('parquet')\
            .mode('append')\
            .option("path", "abfss://silver@<datalake_name>.dfs.core.windows.net/AdventureWorks_Sales")\
            .save()

In [ ]:
df_sales.show(5)